# Agent: Registry Control (Phase 7)

This notebook is the source of truth for criminal counts.

It subscribes to camera detection decisions and publishes official updates to:
- `simcity/surveillance/registry/updates`

Rules:
- Increment `criminal_count` only when `detected=true`
- Deduplicate by `event_id`
- Keep processing simple and explicit

## Quick Start Cheat Sheet (Run Order)

Run-All friendly sequence for reliable registry updates:

1. Click **Run All** in dashboard first.
2. Click **Run All** in camera detector.
3. Click **Run All** in this registry notebook.
4. Click **Run All** in humans producer last.
5. Keep cleanup opt-in (`RUN_CLEANUP=False`) until run is finished.

Tip: this notebook increments counts only when detector sends `detected=true`, and deduplicates strictly by `event_id`.

## How this notebook works

- You subscribe to camera detection decisions.
- You validate payload shape and normalize key fields.
- You deduplicate by `event_id` so duplicates do not increment counts.
- You increment `criminal_count` only when `detected=true`.
- You publish canonical registry updates for dashboard coloring.
- You keep stats snapshots so you can verify processing behavior.

In [ ]:
# Cell 2: Imports
from __future__ import annotations

# JSON is used for MQTT payload serialization/deserialization.
import json

# threading + deque support the callback-thread -> inbox processing pattern.
import threading
import time
from collections import deque
from typing import Any

# Project helpers keep config loading and MQTT usage consistent across notebooks.
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

In [ ]:
# Cell 3: Config + MQTT contract constants
# Load shared config so broker host/port and timing can be changed centrally.
cfg = load_config()
sim_cfg = cfg.simulation

# Topic names are fixed by the surveillance contract used by all notebooks.
SURVEILLANCE_TOPIC_ROOT = "simcity/surveillance"
TOPIC_DETECTIONS = f"{SURVEILLANCE_TOPIC_ROOT}/detections/camera"
TOPIC_REGISTRY_UPDATES = f"{SURVEILLANCE_TOPIC_ROOT}/registry/updates"

# Live-stream settings: low latency, no retained history.
LIVE_QOS = 0
LIVE_RETAIN = False

# Registry loop pace follows simulation delay from config (fallback: 1 second/step).
STEP_DELAY_S = float(sim_cfg.step_delay_s) if (sim_cfg is not None and sim_cfg.step_delay_s > 0) else 1.0

print(f"Registry broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
print(f"Topics -> sub: {TOPIC_DETECTIONS}, pub: {TOPIC_REGISTRY_UPDATES}")
print(f"Registry step delay: {STEP_DELAY_S}s")

Registry broker: 127.0.0.1:1883
Topics -> sub: simcity/surveillance/detections/camera, pub: simcity/surveillance/registry/updates
Registry step delay: 1.0s


In [ ]:
# Cell 4: Registry state + local step timing
# Lock protects shared state between MQTT callback thread and processing loop.
state_lock = threading.Lock()

# Inbox decouples fast MQTT receive from slower validation/update processing.
inbox: deque[dict[str, Any]] = deque()

# Local time anchor used to estimate "current step" from elapsed wall time.
anchor_step: int | None = None
anchor_started_at: float | None = None

# Registry source of truth keyed by human_id.
registry: dict[str, dict[str, Any]] = {}

# Prevents counting the same event twice if duplicate MQTT messages arrive.
processed_event_ids: set[str] = set()

# Counters for quick debugging and sanity checks.
stats = {
    "received": 0,
    "processed": 0,
    "incremented": 0,
    "duplicates": 0,
    "dropped_invalid": 0,
    "dropped_out_of_order": 0,
}

# MQTT objects are initialized in the connect/subscription cell.
mqtt_connector: MqttConnector | None = None
mqtt_publisher: MqttPublisher | None = None
subscription_enabled = False

def current_local_step_unlocked() -> int | None:
    # Returns None until the first message defines the anchor step.
    if anchor_step is None or anchor_started_at is None:
        return None
    elapsed = time.perf_counter() - anchor_started_at
    return anchor_step + int(elapsed // STEP_DELAY_S)

In [ ]:
# Cell 5: MQTT subscription callback (queue only)
def enqueue_detection(payload: dict[str, Any]) -> None:
    global anchor_step, anchor_started_at

    with state_lock:
        # First valid payload defines the registry's local step clock.
        if anchor_step is None and "step" in payload:
            try:
                anchor_step = int(payload["step"])
                anchor_started_at = time.perf_counter()
            except Exception:
                # Keep running even if payload step is malformed.
                pass

        # Queue payload for processing in the main loop.
        inbox.append(payload)

def on_detection_message(client, userdata, message):
    # Callback should stay lightweight: decode JSON, count, and enqueue.
    try:
        payload = json.loads(message.payload.decode("utf-8"))
        with state_lock:
            stats["received"] += 1
        enqueue_detection(payload)
    except Exception:
        with state_lock:
            stats["dropped_invalid"] += 1

try:
    # Connect registry notebook to the shared broker with unique client suffix.
    mqtt_connector = MqttConnector(cfg.mqtt, client_id_suffix="registry-control")
    mqtt_connector.connect()

    if mqtt_connector.wait_for_connection(timeout=5.0):
        mqtt_publisher = MqttPublisher(mqtt_connector)
        mqtt_connector.client.on_message = on_detection_message
        mqtt_connector.client.subscribe(TOPIC_DETECTIONS, qos=LIVE_QOS)
        subscription_enabled = True
        print(f"Registry subscribed to {TOPIC_DETECTIONS}")
    else:
        print("Registry MQTT timeout. Running in idle mode.")
except Exception as exc:
    print(f"Registry MQTT unavailable ({exc}). Running in idle mode.")

Registry subscribed to simcity/surveillance/detections/camera


In [ ]:
# Cell 6: Processing loop
def process_detection(payload: dict[str, Any]) -> None:
    # Minimal payload contract required for registry accounting.
    required = ("step", "event_id", "human_id", "detected")
    if not all(field in payload for field in required):
        raise ValueError("Missing required detection fields")

    # Normalize incoming values to expected types.
    step = int(payload["step"])
    event_id = str(payload["event_id"])
    human_id = str(payload["human_id"])
    detected = bool(payload["detected"])

    # Registry contract depends on event_id/human_id/detected; camera_cell is optional here.
    # Deduplicate by event_id so each event increments at most once.
    with state_lock:
        if event_id in processed_event_ids:
            stats["duplicates"] += 1
            return
        processed_event_ids.add(event_id)

    # Only detected=true should affect criminal_count.
    if not detected:
        with state_lock:
            stats["processed"] += 1
        return

    with state_lock:
        # Create entry for first-time human, then increment count.
        record = registry.get(human_id)
        if record is None:
            record = {"name": human_id, "criminal_count": 0}
            registry[human_id] = record
        record["criminal_count"] += 1
        criminal_count = int(record["criminal_count"])
        stats["processed"] += 1
        stats["incremented"] += 1

    if mqtt_publisher is not None:
        # Publish canonical registry state so dashboard can color humans by criminal_count.
        update_payload = {
            "step": step,
            "human_id": human_id,
            "name": record["name"],
            "criminal_count": criminal_count,
        }
        mqtt_publisher.publish_json(
            TOPIC_REGISTRY_UPDATES,
            json.dumps(update_payload),
            qos=LIVE_QOS,
            retain=LIVE_RETAIN,
        )

def run_registry_loop(total_steps: int = 60) -> None:
    print("Registry loop started...")
    for _ in range(total_steps):
        step_started = time.perf_counter()

        # Drain inbox fully each step so backlog does not grow.
        while True:
            with state_lock:
                if not inbox:
                    break
                payload = inbox.popleft()
            try:
                process_detection(payload)
            except Exception:
                # Invalid payloads are counted and skipped.
                with state_lock:
                    stats["dropped_invalid"] += 1

        # Keep loop timing aligned with simulation step duration.
        elapsed = time.perf_counter() - step_started
        time.sleep(max(0.0, STEP_DELAY_S - elapsed))

    print("Registry loop complete.")

In [6]:
# Cell 7: Start registry loop (Run All friendly: keep active while producer runs)
run_registry_loop(total_steps=150)

Registry loop started...
Registry loop complete.


In [ ]:
# Cell 8: Status snapshot
# Take a locked snapshot to avoid printing partially-updated state.
with state_lock:
    stats_snapshot = dict(stats)
    registry_snapshot = dict(registry)

print("Registry status:")
for key, value in stats_snapshot.items():
    print(f"- {key}={value}")

print("\nRegistry entries:")
if registry_snapshot:
    # Sorted output is easier to compare between runs.
    for human_id in sorted(registry_snapshot):
        entry = registry_snapshot[human_id]
        print(f"- {entry['name']}: criminal_count={entry['criminal_count']}")
else:
    print("- none")

Registry status:
- received=147
- processed=147
- incremented=54
- duplicates=0
- dropped_invalid=0
- dropped_out_of_order=0

Registry entries:
- Ava Jensen: criminal_count=4
- Emma Larsen: criminal_count=3
- Freja Rasmussen: criminal_count=2
- Lucas Mortensen: criminal_count=3
- Noah Pedersen: criminal_count=12
- Oliver Christensen: criminal_count=3
- Sofia Madsen: criminal_count=20
- William Thomsen: criminal_count=7


In [ ]:
# Cell 9: Cleanup (opt-in; safe with Run All)
RUN_CLEANUP = False  # Set to True when you want to disconnect

if RUN_CLEANUP:
    # Graceful disconnect avoids stale client sessions on broker.
    if mqtt_connector is not None and mqtt_connector.client.is_connected():
        mqtt_connector.disconnect()
        print("Registry MQTT connector disconnected.")
    else:
        print("Registry cleanup: no active MQTT connection.")
else:
    # Keep connection alive during normal Run All workflow.
    print("Registry cleanup skipped (RUN_CLEANUP=False).")

Registry cleanup skipped (RUN_CLEANUP=False).
